# Exploratory analysis of the regression data

The supplied training data has 53 anonymized numeric predictors and one continuous target. The hidden file contains predictors only. The evaluation metric is RMSE. The hidden file is checked for schema and drift, and its rows are never used to fit the model or choose a candidate.

In [1]:
import os
from pathlib import Path
import numpy as np

from regression import FEATURES, read_csv, split_indices

data_dir = Path(os.environ.get("DS_TASK_DATA", "."))
X, y = read_csv(data_dir / "train.csv", training=True)
hidden, _ = read_csv(data_dir / "hidden_test.csv", training=False)
fit, selection, audit = split_indices(len(y), 42)
print(f"Training shape: {X.shape}; hidden shape: {hidden.shape}")
print(f"Fit rows: {len(fit):,}; selection rows: {len(selection):,}; audit rows: {len(audit):,}")
print(f"Missing cells: training {np.isnan(X).sum():,}; hidden {np.isnan(hidden).sum():,}")
print(f"Unique feature vectors in training: {len(np.unique(X, axis=0)):,}")

Training shape: (90000, 53); hidden shape: (10000, 53)
Fit rows: 72,000; selection rows: 9,000; audit rows: 9,000
Missing cells: training 0; hidden 0
Unique feature vectors in training: 90,000


## Target distribution

The mean minimizes squared error among constant predictors. Compare its errors against feature-dependent candidates on the model-selection split and repeat the comparison on the independent audit split.

In [2]:
fit_y = y[fit]
q = np.quantile(fit_y, [0, 0.01, 0.25, 0.5, 0.75, 0.99, 1])
print("Fit target quantiles (0%, 1%, 25%, 50%, 75%, 99%, 100%):", np.round(q, 3))
print(f"Fit target mean={fit_y.mean():.6f}; population std={fit_y.std():.6f}")
counts, edges = np.histogram(fit_y, bins=10)
for lo, hi, count in zip(edges[:-1], edges[1:], counts):
    print(f"{lo:6.2f} to {hi:6.2f}  {'#' * round(35 * count / counts.max())}  {count:,}")

Fit target quantiles (0%, 1%, 25%, 50%, 75%, 99%, 100%): [3.0000e-03 1.0010e+00 2.5124e+01 5.0040e+01 7.5088e+01 9.9084e+01
 9.9999e+01]
Fit target mean=50.050376; population std=28.891668
  0.00 to  10.00  ##################################  7,254
 10.00 to  20.00  ##################################  7,126
 20.00 to  30.00  ##################################  7,178
 30.00 to  40.00  ##################################  7,171
 40.00 to  50.00  ##################################  7,243
 50.00 to  60.00  ##################################  7,103
 60.00 to  70.00  ##################################  7,167
 70.00 to  80.00  ###################################  7,378
 80.00 to  90.00  ##################################  7,209
 90.00 to 100.00  ##################################  7,171


## Signal and feature drift

Use only the fit data for supervised diagnostics. A Pearson correlation can miss nonlinear structure, so the checks also compare target means across feature quantiles and screen pairwise products. Exploratory maxima across many columns or pairs can occur by chance. The validation splits determine whether any candidate generalizes.

In [3]:
fit_X = X[fit]
fill = np.nan_to_num(np.nanmedian(fit_X, axis=0))
fit_X = np.where(np.isnan(fit_X), fill, fit_X)
center = fit_X.mean(axis=0)
spread = fit_X.std(axis=0)
spread = np.where(spread > 0, spread, 1)
Z = (fit_X - center) / spread
standard_y = (fit_y - fit_y.mean()) / (fit_y.std() or 1)
linear = Z.T @ standard_y / len(fit)
top = np.argsort(np.abs(linear))[-5:][::-1]
print("Largest absolute fit correlations:", [(FEATURES[j], round(float(linear[j]), 5)) for j in top])

effects = []
for j in range(fit_X.shape[1]):
    cuts = np.unique(np.quantile(fit_X[:, j], np.arange(0.1, 1.0, 0.1)))
    assigned = np.searchsorted(cuts, fit_X[:, j], side="right")
    groups = np.bincount(assigned, minlength=len(cuts) + 1)
    means = np.bincount(assigned, weights=fit_y, minlength=len(groups)) / np.maximum(groups, 1)
    effects.append(np.sqrt(np.average((means - fit_y.mean()) ** 2, weights=groups)))
top_bin = np.argsort(effects)[-5:][::-1]
print("Largest target-mean shifts across fit quantiles:", [(FEATURES[j], round(float(effects[j]), 4)) for j in top_bin])

pairs = np.triu(np.abs(Z.T @ (standard_y[:, None] * Z) / len(fit)), k=1)
a, b = np.unravel_index(np.argmax(pairs), pairs.shape)
print(f"Largest standardized-pair association: features {a} and {b}, value {pairs[a, b]:.5f}")

if len(hidden):
    drift = np.abs(np.nanmean(hidden, axis=0) - center) / spread
    top_drift = np.argsort(drift)[-5:][::-1]
    print("Largest fit/hidden mean shifts in fit SD units:", [(FEATURES[j], round(float(drift[j]), 5)) for j in top_drift])

Largest absolute fit correlations: [('7', 0.01136), ('40', -0.00825), ('50', -0.00696), ('35', -0.00632), ('26', 0.00579)]
Largest target-mean shifts across fit quantiles: [('6', 28.3085), ('35', 0.4783), ('7', 0.4681), ('4', 0.4495), ('32', 0.4294)]
Largest standardized-pair association: features 6 and 8, value 0.33303
Largest fit/hidden mean shifts in fit SD units: [('38', 0.03658), ('9', 0.0267), ('7', 0.02476), ('43', 0.02115), ('28', 0.0205)]


## Diagnose the nonlinear signal

The binned model and trees expose a strong non-monotonic association with feature `6`. On the fit rows, rounding its square to the nearest integer yields the integer part of the target. Feature `7` explains the remaining fraction. This motivates the magnitude-and-remainder candidate. The candidate learns coefficients on fit rows, and the selection and audit splits check whether the relationship holds outside those rows.

The relationship is unusually exact. These features could be target-derived in a real data pipeline, so their provenance and availability at prediction time would need verification before production use.

In [4]:
squared = np.square(fit_X[:, 6])
integer_part = np.rint(squared)
remainder = fit_X[:, 7]
reconstructed = integer_part + remainder
print(f"Maximum distance of feature 6 squared from an integer: {np.max(np.abs(squared - integer_part)):.3e}")
print(f"Fit rows reconstructed exactly: {np.count_nonzero(fit_y == reconstructed):,}/{len(fit_y):,}")
print(f"RMSE with the rounded magnitude alone: {np.sqrt(np.mean((fit_y - integer_part) ** 2)):.6f}")
print(f"RMSE after adding feature 7: {np.sqrt(np.mean((fit_y - reconstructed) ** 2)):.12f}")

Maximum distance of feature 6 squared from an integer: 1.421e-14
Fit rows reconstructed exactly: 72,000/72,000
RMSE with the rounded magnitude alone: 0.575627
RMSE after adding feature 7: 0.000000000000


## Model comparison

The candidate list includes a constant model, a regularized linear model, a binned additive model, two boosted-tree configurations, and the magnitude-and-remainder model. A candidate must beat the mean on model-selection data with a conservative paired error margin. The audit data is scored after the decision is fixed. The selected model family is then fit on all labeled rows for submission.

In [5]:
import json

path = Path("regression_metrics.json")
if path.exists():
    report = json.loads(path.read_text(encoding="utf-8"))
    print(f"{'candidate':21} {'selection RMSE':>15} {'audit RMSE':>13}")
    for name, values in report["candidates"].items():
        print(f"{name:21} {values['selection_rmse']:15.6f} {values['audit_rmse']:13.6f}")
    print("Selected:", report["selected"])
else:
    print("Run train.py to generate regression_metrics.json.")

candidate              selection RMSE    audit RMSE
mean                        28.970772     28.866642
ridge                       28.968676     28.867167
binned_ridge                 7.500494      7.600827
lightgbm_small               0.550875      0.573028
lightgbm_flexible            0.196719      0.194906
magnitude_remainder          0.000000      0.000000
Selected: magnitude_remainder
